# Exercise 1

In [ ]:
import matplotlib.pyplot as plt
from enn554.utilities import read_TMY, satellite_image
from numpy import sign, abs
from scipy.optimize import minimize
from enn554.math import sind, cosd, acosd
from enn554.paths import data_dir
dd = data_dir() # points to the data directory where you store your TMY file and will save other things

In [ ]:
tmy_brisbane,metadata = read_TMY(dd/'tutorial_3/Brisbane_TMY.csv',now_year=2026)
lat,lon = metadata['Latitude'], metadata['Longitude']
tmy_brisbane.plot(x='Datetime',y='GHI')

## Compute POA Irradiance

In [ ]:
L,L_st = 360-lon, 360-150
rho = 0.2

doy = tmy_brisbane['Datetime'].dt.day_of_year
delta = 23.45*sind(360*(doy-81)/365) # degrees
B = 360*(doy-81)/364
E = 9.87*sind(2*B) - 7.53*cosd(B) - 1.5*sind(B) # minutes
hod = tmy_brisbane['Datetime'].dt.hour + tmy_brisbane['Datetime'].dt.minute/60 # fractional hour of day
t_solar = hod + 4/60.0*(L-L_st) + E/60.0 # fractional hour of day of solar time
omega = 15*(t_solar-12) # degrees 

zenith = acosd(sind(lat)*sind(delta) + cosd(lat)*cosd(delta)*cosd(omega))
azimuth = sign(omega)*abs( acosd( (cosd(zenith)*sind(lat)-sind(delta))/(sind(zenith)*cosd(lat)) ) )

tmy_brisbane['zenith'] = zenith
tmy_brisbane['azimuth'] = azimuth

In [ ]:
tmy_brisbane

In [ ]:
GHI,DNI = tmy_brisbane['GHI'], tmy_brisbane['DNI']
DHI = GHI - DNI*cosd(zenith)
tmy_brisbane['DHI2'] = DHI
tmy_brisbane

In [ ]:
def POA(tmy,beta,gamma,rho):
    zenith, azimuth = tmy['zenith'], tmy['azimuth']
    cosAOI = sind(zenith)*sind(beta)*cosd(azimuth-gamma) + cosd(zenith)*cosd(beta)
    cosAOI[cosAOI<0] = 0
    poa = DNI*cosAOI + DHI*(1+cosd(beta))/2 + GHI*rho*(1-cosd(beta))/2

    return poa

tmy_brisbane['POA_flat'] = POA(tmy_brisbane,beta=0,gamma=180,rho=rho)
insolation = sum(tmy_brisbane['POA_flat'])/1000 # kWh/m^2/year

ax = tmy_brisbane.plot(x='Datetime',y='POA_flat')
ax.set_ylabel('POA Irradiance (W/m2)')
ax.set_title(f'POA Irradiance for Flat Panel, Insolation={insolation:.1f} kWh/m2/year')
ax.set_xlabel('Datetime')